
# BM Supermercados and Consum API Client

This notebook collects product and price information from the BM and Consum JSON APIs.

## Collection process

1. Searches for the product terms defined in `SEARCH_TERMS`.
2. Retrieves all available pages with retries.
3. Extracts product IDs, EANs, brands, prices, offers, quantities, and timestamps.
4. Preserves the original API price fields in RAW format.
5. Validates the completeness of both supermarket collections.
6. Saves current RAW files and dated historical RAW snapshots.

## Outputs

### BM
- Current RAW: `data/raw/bm_products_raw.csv`
- Historical RAW: `data/history/bm/YYYY-MM-DD_raw.csv`

### Consum
- Current RAW: `data/raw/consum_products_raw.csv`
- Historical RAW: `data/history/consum/YYYY-MM-DD_raw.csv`

The central pipeline handles price normalization, promotion validation, deduplication, classification, and historical analysis.

## Collection policy

- Both supermarket collections must complete successfully before saving.
- Existing historical snapshots are never overwritten.
- TEST_MODE allows testing selected searches without saving files.


In [ ]:
import re
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests

CHAINS = {
    "BM": "https://www.online.bmsupermercados.es",
    "Consum": "https://tienda.consum.es",
}
SEARCH_PATH = "/api/rest/V1.0/catalog/product"

SEARCH_TERMS = [
    "leche", "huevos", "manzana", "plátano", "tomate", "arroz",
    "pasta", "pan", "pollo", "carne picada", "salmón", "yogur",
    "queso", "agua", "zumo", "atún", "garbanzos", "aceite",
    "detergente", "papel higiénico"
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------
TEST_MODE = False
TEST_TERMS = ["leche", "yogur", "garbanzos", "detergente"]

TERMS_TO_FETCH = TEST_TERMS if TEST_MODE else SEARCH_TERMS

PAGE_SIZE = 100          # maximum the API accepts
REQUEST_PAUSE = 1.0      # seconds between requests
MAX_RETRIES = 3
TIMEOUT = 20
API_RESULT_CAP = 500     # the API never returns more than 500 results per search

HEADERS = {
    "User-Agent": "UC3M-WebAnalytics-StudentProject/1.0 (academic, non-commercial)",
    "Accept": "application/json",
}

print("Searches to run:", TERMS_TO_FETCH)

Searches to run: ['leche', 'yogur', 'garbanzos', 'detergente']


In [2]:
# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------


def find_project_root():
    current = Path.cwd().resolve()

    for path in (current, *current.parents):
        if (
            (path / "api_clients").is_dir()
            and (path / "src").is_dir()
            and (path / "data").is_dir()
        ):
            return path

    raise FileNotFoundError(
        "Could not locate the project root."
    )



BASE_DIR = find_project_root()

RAW_DIR = BASE_DIR / "data" / "raw"
HISTORY_DIR = BASE_DIR / "data" / "history"

RUN_DATE = datetime.now().strftime("%Y-%m-%d")

RAW_PATHS = {
    supermarket: RAW_DIR / f"{supermarket.lower()}_products_raw.csv"
    for supermarket in CHAINS
}

RAW_HISTORY_PATHS = {
    supermarket: (
        HISTORY_DIR / supermarket.lower() / f"{RUN_DATE}_raw.csv"
    )
    for supermarket in CHAINS
}

print("Project:", BASE_DIR)
print("Collection date:", RUN_DATE)

for supermarket in CHAINS:
    print(f"\n{supermarket}")
    print("Current RAW:", RAW_PATHS[supermarket])
    print("Historical RAW:", RAW_HISTORY_PATHS[supermarket])

Project: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Collection date: 2026-11-20

BM
Current RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\bm_products_raw.csv
Historical RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\bm\2026-11-20_raw.csv

Consum
Current RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\consum_products_raw.csv
Historical RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\consum\2026-11-20_raw.csv


## API requests

`get_page` makes one request with retries (exponential backoff on network errors,
429 or 5xx). `search_all` walks through every page of a term until `hasMore` is
false.

In [3]:
session = requests.Session()
session.headers.update(HEADERS)


def get_page(base_url, term, offset):
    url = base_url + SEARCH_PATH
    params = {"q": term, "limit": PAGE_SIZE, "offset": offset}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(url, params=params, timeout=TIMEOUT)

            if response.status_code == 429 or response.status_code >= 500:
                raise requests.HTTPError(f"HTTP {response.status_code}")

            response.raise_for_status()
            return response.json()

        except (requests.RequestException, ValueError) as exc:
            if attempt == MAX_RETRIES:
                raise
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({exc}). Retrying in {wait} s...")
            time.sleep(wait)


def search_all(base_url, term):
    products = []
    offset = 0
    total = None

    while True:
        data = get_page(base_url, term, offset)
        total = data.get("totalCount", 0)
        page = data.get("products", [])
        products.extend(page)

        time.sleep(REQUEST_PAUSE)

        if not data.get("hasMore") or not page:
            break
        offset += PAGE_SIZE

    return products, total

## Flatten the JSON

Each product is a nested JSON object. We keep the useful fields:

- `priceData.prices`: a list with `PRICE` (regular price) and, when there is a
  promotion, `OFFER_PRICE`. Despite its name, `centAmount` is already **in euros**.
- `centUnitAmount`: price per reference unit, described in `unitPriceUnitType`
  (BM uses `LITRO`/`KILO`; Consum uses `1 L`, `1 Kg`...).

In [4]:
def get_price(prices, price_id):
    for price in prices:
        if price.get("id") == price_id:
            value = price.get("value", {})
            return value.get("centAmount"), value.get("centUnitAmount")
    return None, None


def flatten_product(product, supermarket, searched_item, scraped_at):
    data = product.get("productData", {})
    price_data = product.get("priceData", {})
    prices = price_data.get("prices", [])

    regular_price, regular_unit_price = get_price(prices, "PRICE")
    offer_price, offer_unit_price = get_price(prices, "OFFER_PRICE")

    categories = product.get("categories") or []

    return {
        "searched_item": searched_item,
        "supermarket": supermarket,
        "product_id": product.get("code"),
        "ean": product.get("ean"),
        "brand": (data.get("brand") or {}).get("name"),
        "name": data.get("name"),
        "description": data.get("description"),
        "category": categories[0].get("name") if categories else None,
        "regular_price": regular_price,
        "offer_price": offer_price,
        "regular_unit_price": regular_unit_price,
        "offer_unit_price": offer_unit_price,
        "price_unit_type": price_data.get("priceUnitType"),
        "unit_price_unit_type": price_data.get("unitPriceUnitType"),
        "available": data.get("availability") == 1
                     and not data.get("temporaryOutOfStock", False),
        "url": data.get("url"),
        "image_url": data.get("imageURL"),
        "scraped_at": scraped_at,
    }

## Single-search test

Before running every search, we check with `leche` on both chains that the number of
rows matches the `totalCount` returned by the API.

In [5]:
# --------------------------------------------------
# OPTIONAL API CONNECTION TEST
# --------------------------------------------------

# Skip this diagnostic during full collections.
# Failures here must not stop the collection process.

if TEST_MODE:

    for supermarket, base_url in CHAINS.items():

        print(f"\nTesting API connection: {supermarket}")

        try:
            products, total = search_all(
                base_url,
                "leche"
            )

            scraped_at = datetime.now().isoformat(
                timespec="seconds"
            )

            df_test = pd.DataFrame([
                flatten_product(
                    product,
                    supermarket,
                    "leche",
                    scraped_at
                )
                for product in products
            ])

            print(
                f"{supermarket}: "
                f"totalCount={total}, "
                f"rows={len(df_test)}"
            )

            if not df_test.empty:
                display(
                    df_test[
                        [
                            "name",
                            "brand",
                            "regular_price",
                            "offer_price",
                            "regular_unit_price",
                            "unit_price_unit_type"
                        ]
                    ].head(5)
                )

        except Exception as exc:

            print(
                f"Diagnostic failed for {supermarket}: "
                f"{type(exc).__name__}: {exc}"
            )

            print(
                "The main collection will still run."
            )



Testing API connection: BM
BM: totalCount=306, rows=306


,name,brand,regular_price,offer_price,regular_unit_price,unit_price_unit_type
0,Leche entera 1 l,BM,0.96,NaN,0.96,LITRO
1,Leche entera 1 l,Ram,1.15,1.04,1.15,LITRO
2,Leche entera 1 l,Pascual,1.39,NaN,1.39,LITRO
3,"Leche entera 1,5 l",BM,1.56,NaN,1.04,LITRO
4,"Leche entera 1,5 l",Central Lechera Asturiana,1.97,NaN,1.32,LITRO



Testing API connection: Consum
Consum: totalCount=303, rows=303


,name,brand,regular_price,offer_price,regular_unit_price,unit_price_unit_type
0,Leche Semidesnatada Brik,CONSUM,0.83,NaN,0.83,1 L
1,Leche Entera Brik,CONSUM,0.96,NaN,0.96,1 L
2,Leche Calcio Semidesnatada Brik,CONSUM,1.05,NaN,1.05,1 L
3,Leche Calcio Desnatada Brik,PULEVA,1.59,1.29,1.59,1 L
4,Leche sin Lactosa Entera Brik,CONSUM,1.03,NaN,1.03,1 L


## All searches

If a search still fails after the retries, it is logged as `ERROR` in the summary
and the loop moves on to the next one.

In [6]:

# --------------------------------------------------
# COLLECT PRODUCTS WITH DEFERRED RETRIES
# --------------------------------------------------

# One initial collection pass + two additional retry passes.
MAX_ADDITIONAL_ATTEMPTS = 2
MAX_SEARCH_PASSES = 1 + MAX_ADDITIONAL_ATTEMPTS

SUCCESS_STATUSES = {"OK", "OK (500 cap)"}

# Define every search once, preserving the original order.
search_jobs = [
    (supermarket, base_url, term)
    for supermarket, base_url in CHAINS.items()
    for term in TERMS_TO_FETCH
]

# Store only successful results.
successful_results = {}

# Keep one final summary record per supermarket and term.
summary_records = {}


# --------------------------------------------------
# EXECUTE COLLECTION PASSES
# --------------------------------------------------

for pass_number in range(1, MAX_SEARCH_PASSES + 1):

    # Only retry searches that have not succeeded.
    pending_jobs = [
        job
        for job in search_jobs
        if (job[0], job[2]) not in successful_results
    ]

    if not pending_jobs:
        print("\nAll searches completed successfully.")
        break

    print("\n" + "=" * 70)
    print(
        f"COLLECTION PASS {pass_number}/{MAX_SEARCH_PASSES}"
    )
    print("Pending searches:", len(pending_jobs))
    print("=" * 70)

    for i, (supermarket, base_url, term) in enumerate(
        pending_jobs,
        start=1
    ):

        key = (supermarket, term)

        print(
            f"\n[{i}/{len(pending_jobs)}] "
            f"{supermarket} - {term} "
            f"(attempt {pass_number})"
        )

        started = time.monotonic()

        rows = []
        total = None
        status = "ERROR"
        error_message = None

        try:

            # Restart the complete search from page 1.
            products, total = search_all(
                base_url,
                term
            )

            scraped_at = datetime.now().isoformat(
                timespec="seconds"
            )

            rows = [
                flatten_product(
                    product,
                    supermarket,
                    term,
                    scraped_at
                )
                for product in products
            ]

            # ------------------------------------------
            # VALIDATE SEARCH COMPLETENESS
            # ------------------------------------------

            if not rows:

                status = "INCOMPLETE"
                error_message = "No products returned."

            elif total is None:

                status = "INCOMPLETE"
                error_message = "Missing total count."

            elif (
                total >= API_RESULT_CAP
                and len(rows) == API_RESULT_CAP
            ):

                status = "OK (500 cap)"

            elif len(rows) == total:

                status = "OK"

            else:

                status = "INCOMPLETE"

                error_message = (
                    f"Expected {total} products, "
                    f"retrieved {len(rows)}."
                )

            # ------------------------------------------
            # KEEP ONLY COMPLETE SEARCH RESULTS
            # ------------------------------------------

            if status in SUCCESS_STATUSES:

                successful_results[key] = pd.DataFrame(
                    rows
                )

                print(
                    f"SUCCESS: {len(rows)} products "
                    f"collected."
                )

            else:

                print(
                    f"INCOMPLETE: {error_message}"
                )

        except Exception as exc:

            status = "ERROR"

            error_message = (
                f"{type(exc).__name__}: {exc}"
            )

            print(
                f"FAILED: {error_message}"
            )

            print(
                "Continuing with the next search."
            )

        elapsed = round(
            time.monotonic() - started,
            2
        )

        # Replace the previous summary for this search.
        # This prevents duplicate summary entries.
        summary_records[key] = {
            "supermarket": supermarket,
            "searched_item": term,
            "total_count": total,
            "rows_extracted": len(rows),
            "status": status,
            "attempts": pass_number,
            "seconds": elapsed,
            "error": error_message,
        }

    # ----------------------------------------------
    # SUMMARY AFTER EACH PASS
    # ----------------------------------------------

    remaining = [
        job
        for job in search_jobs
        if (job[0], job[2]) not in successful_results
    ]

    print("\nPASS SUMMARY")
    print("Successful searches:", len(successful_results))
    print("Remaining failures:", len(remaining))

    if remaining and pass_number < MAX_SEARCH_PASSES:

        print("\nFailed searches to retry:")

        for supermarket, _, term in remaining:
            print(f"  - {supermarket}: {term}")

        # Give the APIs a short pause before retrying.
        time.sleep(10)


# --------------------------------------------------
# BUILD FINAL RESULTS
# --------------------------------------------------

# Include each successful search exactly once.
# Keep the structure expected by the next notebook cell.

all_results = {
    supermarket: [
        successful_results[(supermarket, term)]
        for term in TERMS_TO_FETCH
        if (supermarket, term) in successful_results
    ]
    for supermarket in CHAINS
}


# Keep exactly one summary row per original search.
summary = [
    summary_records[(supermarket, term)]
    for supermarket in CHAINS
    for term in TERMS_TO_FETCH
]

df_summary = pd.DataFrame(summary)


# --------------------------------------------------
# FINAL COLLECTION REPORT
# --------------------------------------------------

print("\n" + "=" * 70)
print("FINAL COLLECTION SUMMARY")
print("=" * 70)

display(df_summary)

print("\nSearch status by supermarket:")

display(
    df_summary.groupby(
        ["supermarket", "status"]
    ).size().rename("searches")
)

failed_searches = df_summary[
    ~df_summary["status"].isin(SUCCESS_STATUSES)
]

recovered_searches = df_summary[
    df_summary["status"].isin(SUCCESS_STATUSES)
    & df_summary["attempts"].gt(1)
]

print("\nTotal searches:", len(df_summary))
print("Successful searches:", len(df_summary) - len(failed_searches))
print("Recovered after retry:", len(recovered_searches))
print("Final failures:", len(failed_searches))

if not recovered_searches.empty:

    print("\nRECOVERED SEARCHES:")

    display(
        recovered_searches[
            [
                "supermarket",
                "searched_item",
                "attempts",
                "rows_extracted"
            ]
        ]
    )

if not failed_searches.empty:

    print("\nWARNING: Some searches remain incomplete.")

    display(
        failed_searches[
            [
                "supermarket",
                "searched_item",
                "status",
                "attempts",
                "error"
            ]
        ]
    )

    print(
        "The validation step will prevent "
        "incomplete data from being published."
    )



COLLECTION PASS 1/3
Pending searches: 8

[1/8] BM - leche (attempt 1)
SUCCESS: 306 products collected.

[2/8] BM - yogur (attempt 1)
SUCCESS: 298 products collected.

[3/8] BM - garbanzos (attempt 1)
SUCCESS: 25 products collected.

[4/8] BM - detergente (attempt 1)
SUCCESS: 75 products collected.

[5/8] Consum - leche (attempt 1)
SUCCESS: 303 products collected.

[6/8] Consum - yogur (attempt 1)
SUCCESS: 187 products collected.

[7/8] Consum - garbanzos (attempt 1)
SUCCESS: 28 products collected.

[8/8] Consum - detergente (attempt 1)
SUCCESS: 98 products collected.

PASS SUMMARY
Successful searches: 8
Remaining failures: 0

All searches completed successfully.

FINAL COLLECTION SUMMARY


,supermarket,searched_item,total_count,rows_extracted,status,attempts,seconds,error
0,BM,leche,306,306,OK,1,6.16,None
1,BM,yogur,298,298,OK,1,4.56,None
2,BM,garbanzos,25,25,OK,1,1.30,None
3,BM,detergente,75,75,OK,1,1.39,None
4,Consum,leche,303,303,OK,1,5.48,None
5,Consum,yogur,187,187,OK,1,3.16,None
6,Consum,garbanzos,28,28,OK,1,1.39,None
7,Consum,detergente,98,98,OK,1,2.09,None



Search status by supermarket:


supermarket  status
BM           OK        4
Consum       OK        4
Name: searches, dtype: int64


Total searches: 8
Successful searches: 8
Recovered after retry: 0
Final failures: 0


In [7]:

# --------------------------------------------------
# PREPARE RAW DATA (one dataframe per chain)
# --------------------------------------------------

raw = {}

for supermarket, frames in all_results.items():
    if not frames:
        print(f"{supermarket}: no data collected.")
        continue

    raw[supermarket] = pd.concat(
        frames, ignore_index=True
    )

    print(
        f"{supermarket}: "
        f"{len(raw[supermarket])} raw rows ready."
    )


BM: 704 raw rows ready.
Consum: 616 raw rows ready.


In [8]:
# --------------------------------------------------
# VALIDATE AND SAVE BM AND CONSUM RAW DATA
# --------------------------------------------------

import os
import tempfile

if TEST_MODE:
    print("TEST MODE: no files will be saved.")

else:
    # --------------------------------------------------
    # VALIDATE COLLECTION COMPLETENESS
    # --------------------------------------------------

    if set(raw) != set(CHAINS):
        raise RuntimeError(
            "Missing supermarket data. No files saved."
        )

    expected_searches = len(CHAINS) * len(SEARCH_TERMS)

    if len(df_summary) != expected_searches:
        raise RuntimeError(
            "Not all expected searches were attempted. "
            "No files saved."
        )

    successful_statuses = ["OK", "OK (500 cap)"]

    if not df_summary["status"].isin(successful_statuses).all():
        raise RuntimeError(
            "Incomplete or failed searches. No files saved."
        )

    for supermarket in CHAINS:
        terms = df_summary.loc[
            df_summary["supermarket"] == supermarket,
            "searched_item"
        ]

        if (
            len(terms) != len(SEARCH_TERMS)
            or set(terms) != set(SEARCH_TERMS)
        ):
            raise RuntimeError(
                f"{supermarket}: missing or repeated searches. "
                "No files saved."
            )

    # --------------------------------------------------
    # VALIDATE RAW DATA
    # --------------------------------------------------

    required_columns = [
        "searched_item",
        "supermarket",
        "product_id",
        "ean",
        "brand",
        "name",
        "regular_price",
        "offer_price",
        "regular_unit_price",
        "offer_unit_price",
        "unit_price_unit_type",
        "scraped_at",
    ]

    for supermarket in CHAINS:
        df_raw = raw[supermarket]

        if df_raw.empty:
            raise RuntimeError(
                f"{supermarket}: empty RAW dataset. "
                "No files saved."
            )

        missing_columns = [
            col for col in required_columns
            if col not in df_raw.columns
        ]

        if missing_columns:
            raise RuntimeError(
                f"{supermarket}: missing RAW columns "
                f"{missing_columns}"
            )

        regular_prices = pd.to_numeric(
            df_raw["regular_price"],
            errors="coerce"
        )

        offer_prices = pd.to_numeric(
            df_raw["offer_price"],
            errors="coerce"
        )

        # Use a positive offer price when available.
        effective_prices = offer_prices.where(
            offer_prices.gt(0),
            regular_prices
        )

        valid_names = (
            df_raw["name"].notna()
            & df_raw["name"].astype(str).str.strip().ne("")
        )

        valid_prices = (
            effective_prices.notna()
            & effective_prices.gt(0)
        )

        if not (valid_names & valid_prices).any():
            raise RuntimeError(
                f"{supermarket}: no valid products found. "
                "No files saved."
            )

        invalid_rows = int(
            (~(valid_names & valid_prices)).sum()
        )

        print(
            f"{supermarket}: {len(df_raw)} RAW rows, "
            f"{invalid_rows} rows requiring validation."
        )

    # --------------------------------------------------
    # PROTECT EXISTING HISTORICAL SNAPSHOTS
    # --------------------------------------------------

    for supermarket in CHAINS:
        path = RAW_HISTORY_PATHS[supermarket]

        if path.exists():
            raise FileExistsError(
                f"{supermarket}: historical RAW already exists "
                f"for {RUN_DATE}: {path}. "
                "No files were modified."
            )

    # --------------------------------------------------
    # PREPARE TEMPORARY FILES
    # --------------------------------------------------

    def prepare_raw_temp(frame, destination):
        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        fd, temp_name = tempfile.mkstemp(
            prefix=".api_raw_",
            suffix=".tmp",
            dir=destination.parent
        )

        try:
            with os.fdopen(
                fd,
                "w",
                encoding="utf-8-sig",
                newline=""
            ) as file:
                frame.to_csv(file, index=False)

            return Path(temp_name)

        except Exception:
            Path(temp_name).unlink(missing_ok=True)
            raise

    history_temps = {}
    current_temps = {}
    created_histories = []

    try:
        for supermarket in CHAINS:
            history_temps[supermarket] = prepare_raw_temp(
                raw[supermarket],
                RAW_HISTORY_PATHS[supermarket]
            )

            current_temps[supermarket] = prepare_raw_temp(
                raw[supermarket],
                RAW_PATHS[supermarket]
            )

        # Create both historical files without overwriting.
        for supermarket in CHAINS:
            history_path = RAW_HISTORY_PATHS[supermarket]

            with history_temps[supermarket].open("rb") as source:
                with history_path.open("xb") as target:
                    created_histories.append(history_path)

                    while True:
                        chunk = source.read(1024 * 1024)

                        if not chunk:
                            break

                        target.write(chunk)

            print(
                f"{supermarket}: historical RAW saved:",
                history_path
            )

        # Update current RAW files only after both histories exist.
        for supermarket in CHAINS:
            os.replace(
                current_temps[supermarket],
                RAW_PATHS[supermarket]
            )

            print(
                f"{supermarket}: current RAW updated:",
                RAW_PATHS[supermarket]
            )

    except Exception:
        print(
            "ERROR: RAW saving was interrupted. "
            "Review current and historical files before retrying."
        )
        raise

    finally:
        for temp_path in [
            *history_temps.values(),
            *current_temps.values()
        ]:
            temp_path.unlink(missing_ok=True)

    # --------------------------------------------------
    # COLLECTION SUMMARY
    # --------------------------------------------------

    print("\nCOLLECTION COMPLETED")
    print("Date:", RUN_DATE)
    print("Status: COMPLETE")

    for supermarket in CHAINS:
        df_raw = raw[supermarket]

        print(f"\n{supermarket}")
        print("RAW rows:", len(df_raw))
        print(
            "Distinct product IDs:",
            df_raw["product_id"].nunique()
        )
        print("Current RAW:", RAW_PATHS[supermarket])
        print("Historical RAW:", RAW_HISTORY_PATHS[supermarket])

TEST MODE: no files will be saved.
